In [12]:
# enum Category { Ring = 0, Chain = 1, Bangle = 2 }
categories = ["Ring", "Chain", "Bangle"]
print(categories[1])   # Chain
import torch
import torch.nn as nn
scores = torch.tensor([2.0, 0.5, -1.0])     # Ring, Chain, Bangle
probs = torch.softmax(scores, dim=0)
print(probs)
print("total:", probs.sum())
print("prediction:", categories[probs.argmax()])

Chain
tensor([0.7856, 0.1753, 0.0391])
total: tensor(1.)
prediction: Ring


In [13]:
loss_fn = nn.CrossEntropyLoss()

correct = torch.tensor([0])                    # the true answer is Ring (index 0)

good_guess = torch.tensor([[3.0, 0.1, -1.0]])  # high score for Ring
bad_guess  = torch.tensor([[-1.0, 3.0, 0.1]])  # high score for Chain (wrong)

print("good guess loss:", loss_fn(good_guess, correct).item())
print("bad guess loss:",  loss_fn(bad_guess, correct).item())

good guess loss: 0.07077422738075256
bad guess loss: 4.070774078369141


In [14]:
torch.manual_seed(0)
categories = ["Ring", "Chain", "Bangle"]
n = 300   # items per category

def make(n, w_min, w_max, l_min, l_max, label):
    w = torch.rand(n) * (w_max - w_min) + w_min    # random weights in the range
    l = torch.rand(n) * (l_max - l_min) + l_min    # random lengths in the range
    x = torch.stack([w, l], dim=1)                 # join into 2 columns
    y = torch.full((n,), label)                    # n copies of the label
    return x, y

x_ring,   y_ring   = make(n, 2, 8,   15, 22,  0)
x_chain,  y_chain  = make(n, 8, 30,  400, 600, 1)
x_bangle, y_bangle = make(n, 15, 40, 55, 70,  2)

x = torch.cat([x_ring, x_chain, x_bangle])   # 900 rows, 2 columns
y = torch.cat([y_ring, y_chain, y_bangle])   # 900 labels

print(x.shape, y.shape)
print(x[:3], y[:3])

torch.Size([900, 2]) torch.Size([900])
tensor([[ 4.9775, 16.8114],
        [ 6.6093, 19.1288],
        [ 2.5309, 16.6819]]) tensor([0, 0, 0])


In [15]:
# 1. Shuffle
perm = torch.randperm(len(x))   # a random order of 0..899
x = x[perm]
y = y[perm]

# 2. Split 80% train / 20% test
split = int(0.8 * len(x))       # 720
x_train, x_test = x[:split], x[split:]
y_train, y_test = y[:split], y[split:]

# 3. Standardise (use the TRAIN mean/std only — no data leakage)
x_mean = x_train.mean(dim=0)
x_std  = x_train.std(dim=0)
x_train_n = (x_train - x_mean) / x_std
x_test_n  = (x_test  - x_mean) / x_std

print("train:", x_train_n.shape, "test:", x_test_n.shape)
print("train count per category:", torch.bincount(y_train))
print("test count per category: ", torch.bincount(y_test))

train: torch.Size([720, 2]) test: torch.Size([180, 2])
train count per category: tensor([237, 237, 246])
test count per category:  tensor([63, 63, 54])


In [16]:
import torch
import torch.nn as nn
class CategoryModel(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.hidden = nn.Linear(2, hidden_size)
        self.relu = nn.ReLU()
        self.output = nn.Linear(hidden_size, 3)   # 3 categories → 3 scores

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

In [17]:
torch.manual_seed(0)
model = CategoryModel(16)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for r in range(1000):
    scores = model(x_train_n)            # shape (720, 3)
    loss = loss_fn(scores, y_train)      # compare with true labels
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if r % 200 == 0:
        print(f"step {r}  loss = {loss.item():.4f}")

step 0  loss = 1.0272
step 200  loss = 0.0059
step 400  loss = 0.0017
step 600  loss = 0.0008
step 800  loss = 0.0004


In [18]:

with torch.no_grad():
    test_scores = model(x_test_n)            # (180, 3)
    preds = test_scores.argmax(dim=1)        # best category for each row
    correct = (preds == y_test)              # True/False for each row
    accuracy = correct.float().mean()

print("test accuracy:", round(accuracy.item() * 100, 1), "%")
print("first 10 predictions:", preds[:10])
print("first 10 true labels:", y_test[:10])

test accuracy: 100.0 %
first 10 predictions: tensor([2, 2, 1, 2, 1, 1, 0, 0, 0, 2])
first 10 true labels: tensor([2, 2, 1, 2, 1, 1, 0, 0, 0, 2])


In [19]:
def predict(weight, length):
    item = torch.tensor([[weight, length]])
    item_n = (item - x_mean) / x_std          # same standardisation as training
    with torch.no_grad():
        probs = torch.softmax(model(item_n), dim=1)[0]
    for name, p in zip(categories, probs):
        print(f"  {name:7s} {p.item()*100:5.1f}%")
    print("→", categories[probs.argmax()])

predict(5, 18)      # small and short
predict(25, 500)    # medium weight, very long
predict(20, 60)

  Ring    100.0%
  Chain     0.0%
  Bangle    0.0%
→ Ring
  Ring      0.0%
  Chain   100.0%
  Bangle    0.0%
→ Chain
  Ring      0.0%
  Chain     0.0%
  Bangle   99.9%
→ Bangle


In [20]:
def accuracy_with_shuffled(col):
    x_broken = x_test_n.clone()                    # copy, don't touch the original
    perm = torch.randperm(len(x_broken))
    x_broken[:, col] = x_broken[perm, col]         # mix up one column only
    with torch.no_grad():
        preds = model(x_broken).argmax(dim=1)
    return round((preds == y_test).float().mean().item() * 100, 1)

print("normal accuracy:          100.0 %")
print("weight column shuffled:  ", accuracy_with_shuffled(0), "%")
print("length column shuffled:  ", accuracy_with_shuffled(1), "%")

normal accuracy:          100.0 %
weight column shuffled:   67.2 %
length column shuffled:   51.1 %


In [21]:
# keep only column 0 (weight)
xw_train = x_train_n[:, :1]     # shape (720, 1)
xw_test  = x_test_n[:, :1]

torch.manual_seed(0)
model_w = nn.Sequential(
    nn.Linear(1, 16),           # 1 input now, not 2
    nn.ReLU(),
    nn.Linear(16, 3)
)
optimizer = torch.optim.Adam(model_w.parameters(), lr=0.01)

for r in range(1000):
    loss = loss_fn(model_w(xw_train), y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    preds = model_w(xw_test).argmax(dim=1)
    acc = (preds == y_test).float().mean()
print("weight-only accuracy:", round(acc.item() * 100, 1), "%")

weight-only accuracy: 81.7 %


In [22]:
a = torch.tensor([48., 50., 52., 50.])
b = torch.tensor([10., 90., 20., 80.])
print("A mean:", a.mean().item(), " std:", a.std().item())
print("B mean:", b.mean().item(), " std:", b.std().item())

A mean: 50.0  std: 1.632993221282959
B mean: 50.0  std: 40.8248291015625
